In [18]:
import os 
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

model = init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000212D88BB4D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000212D88BBED0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [19]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str= Field(description="The title of the movie") # llm will generate the response based on the filed being given
    year:int= Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")
    

In [20]:
model_with_structured = model.with_structured_output(Movie)
model_with_structured
# PydanticToolsParser(first_tool_only=True, tools=[<class '__main__.Movie

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000212D88BB4D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000212D88BBED0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The ti

In [21]:
model.invoke("Provide details around the movie inception")

AIMessage(content='**Inception (2010) – Quick Reference Guide**\n\n| Item | Details |\n|------|---------|\n| **Title** | *Inception* |\n| **Release Year** | 2010 |\n| **Genre** | Science‑fiction, Action, Thriller, Heist |\n| **Running Time** | 148 minutes |\n| **Director / Writer** | Christopher Nolan |\n| **Producers** | Emma Thomas, Christopher Nolan, Michael Caveney (executive) |\n| **Music** | Hans Zimmer |\n| **Cinematography** | Wally Pfister |\n| **Editor** | Lee Smith |\n| **Production Companies** | Syncopy, Legendary Pictures, Warner Bros. Pictures |\n| **Distributor** | Warner Bros. Pictures |\n| **Budget** | ≈\u202f$160\u202fmillion (USD) |\n| **Box‑Office Gross** | ≈\u202f$836\u202fmillion worldwide (USD) |\n| **MPAA Rating** | PG‑13 (Violence, some language, brief nudity) |\n\n---\n\n## Main Cast & Characters\n\n| Actor / Actress | Character | Brief Description |\n|-----------------|-----------|-------------------|\n| Leonardo DiCaprio | Dom Cobb | A skilled “extractor” wh

In [ ]:
model_with_structured.invoke("Provide details around the movie inception")
# If title will be having an int then it will give use result as error

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [23]:
## Message output alongside Parsed structure 

from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str= Field(..., description="The title of the movie") # llm will generate the response based on the filed being given
    year:int= Field(...,description="This year the movie was released")
    director:str=Field(...,description="The director of the movie")
    rating:float=Field(...,description="The movie rating out of 10")

model_with_structured = model.with_structured_output(Movie, include_raw=True)
model_with_structured.invoke("Provide details around the movie inception")
# If title will be having an int then it will give use result as error

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Provide details around the movie inception". They likely want details like director, rating, title, year. We can call the function Movie with appropriate parameters. The function expects director, rating, title, year. Provide those.\n\nInception details: Director Christopher Nolan, rating maybe 8.8 (IMDb) or 8.8 out of 10. Title "Inception". Year 2010.\n\nWe\'ll call the function.', 'tool_calls': [{'id': 'fc_19638eab-e76d-40ec-a7f8-cb6c599c374d', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 144, 'prompt_tokens': 157, 'total_tokens': 301, 'completion_time': 0.30130399, 'completion_tokens_details': {'reasoning_tokens': 92}, 'prompt_time': 0.006682994, 'prompt_tokens_details': None, 'queue_time': 0.39940697, 'total_time': 0.307986984}, 'model_name

Nested Structure

In [34]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int 
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in Million USD") # Putting None as default value in Budget

model_with_structured_output = model.with_structured_output(MovieDetails)
response = model_with_structured_output.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Miles'), Actor(name='Cillian Murphy', role='Robert Fischer')], genres=['Action', 'Adventure', 'Sci-Fi', 'Thriller'], budget=160000000.0)

TypeDict

Here the input validation will not happen like it happen in pydantic, even if int in director name, then also no error

In [36]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    title: Annotated[str,..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_with_structured_output = model.with_structured_output(MovieDict)
response = model_with_structured_output.invoke("Provide details about the movie Avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [37]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int 
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in Million USD") # Putting None as default value in Budget

model_with_structured_output = model.with_structured_output(MovieDetails)
response = model_with_structured_output.invoke("Provide details about the movie Inception")
response

{'budget': 160000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Elliot Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Eames'},
  {'name': 'Ken Watanabe', 'role': 'Saito'},
  {'name': 'Marion Cotillard', 'role': 'Mal'},
  {'name': 'Cillian Murphy', 'role': 'Robert Fischer'},
  {'name': 'Michael Caine', 'role': 'Professor Stephen Miles'},
  {'name': 'Dileep Rao', 'role': 'Yusuf'},
  {'name': 'Tom Berenger', 'role': 'Bob'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi', 'Thriller'],
 'title': 'Inception',
 'year': 2010}

In [ ]:
model.profile # For model with Sturtured output, we donot see the profile 

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

DataClasses

Typically containing mainly data, although there aren't really any instructions. You can create it using the @dataclass decorators 


In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """ Contact info for a person"""
    name : str
    email: str
    phone : str

agent = create_agent(
    model=model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role":"user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})
# Extract the data from the content 
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')